# EcoRewind 2.0 — amendment A3: matched placebo lead-0 composites, then baselines v3 (Colab, **CPU runtime**)

The event lead 0 is a composite of the ~1 month after landfall; until now the placebo lead 0 was a full quarter.
This exports placebo composites over the **same days** (`*_PPOST`, 12 small files, a few EECU-h) and re-runs the
baselines so event and placebo lead 0 are compared like for like.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
import os
TOKEN = userdata.get('GITHUB_TOKEN')
REPO = '/content/ecorewind2'
if os.path.isdir(REPO):
    !git -C {REPO} pull -q
else:
    !git clone -q https://{TOKEN}@github.com/DAISHINKAN7/EcoRewind2.git {REPO}
%cd {REPO}
!git log --oneline -1
!pip -q install -r requirements.txt earthengine-api
D = '/content/drive/MyDrive/EcoRewind2'
os.environ['ECOREWIND_REPORTS'] = f'{D}/reports'
PROJECT = 'ecorewind-2026'     # <- same Earth Engine project as Phase 0

### 1. Earth Engine: authenticate, dry run, submit (12 tasks)

In [ ]:
import ee
ee.Authenticate()
ee.Initialize(project=PROJECT)
for s in ('export_trackA_s2.py', 'export_trackB_hls.py', 'export_s1.py'):
    !python gee/{s} --placebo-post --dry-run --project $PROJECT

In [ ]:
for s in ('export_trackA_s2.py', 'export_trackB_hls.py', 'export_s1.py'):
    !python gee/{s} --placebo-post --project $PROJECT

### 2. Wait until all 12 are COMPLETED (re-run this cell; typically 10–40 min)

In [ ]:
import collections
ts = [t for t in ee.batch.Task.list() if 'PPOST' in t.config.get('description', '')]
print(collections.Counter(t.state for t in ts))
for t in ts:
    if t.state == 'FAILED': print(t.config['description'], t.status().get('error_message'))
!ls -lh /content/drive/MyDrive/EcoRewind2_exports/*PPOST*

### 3. Cubes (restored from Drive) and baselines v3

In [ ]:
# 1. reconnect Drive
from google.colab import drive
drive.flush_and_unmount()
drive.mount('/content/drive', force_remount=True)

# 2. discard anything built from the incomplete copy
!rm -rf /content/ppost /content/tables/*_v3.npz /content/drive/MyDrive/EcoRewind2/runs/phase2_baselines_v3

# 3. copy the 12 placebo files again, retrying each one up to 5 times
import glob, shutil, time, os
os.makedirs('/content/ppost', exist_ok=True)
src = sorted(glob.glob('/content/drive/MyDrive/EcoRewind2_exports/*PPOST*.tif'))
print(len(src), 'files on Drive (need 12)')
for f in src:
    for i in range(5):
        try:
            shutil.copy(f, '/content/ppost/'); break
        except OSError as e:
            print('retry', os.path.basename(f), e); time.sleep(5)
!ls /content/ppost | wc -l

In [ ]:
!python scripts/cubes_cache.py --drive {D}/cubes --local /content/cubes --exports-drive /content/drive/MyDrive/EcoRewind2_exports
!mkdir -p /content/ppost && cp /content/drive/MyDrive/EcoRewind2_exports/*PPOST* /content/ppost/
RUN = 'phase2_baselines_v3'
!python scripts/run_baselines.py --cubes /content/cubes --phase1 {D}/phase1 --out {D}/runs --run-id {RUN} --table-cache /content/tables --ppost-exports /content/ppost

In [ ]:
!git -C /content/ecorewind2 pull -q
!git -C /content/ecorewind2 log --oneline -1    # should show c287d0a
D = '/content/drive/MyDrive/EcoRewind2'
RUN = 'phase2_baselines_v3'
!python scripts/run_baselines.py --cubes /content/cubes --phase1 {D}/phase1 --out {D}/runs --run-id {RUN} --table-cache /content/tables --ppost-exports /content/ppost

In [ ]:
from IPython.display import Markdown
Markdown(open(f'{D}/runs/{RUN}/summary.md').read())

Send me `runs/phase2_baselines_v3/summary.md`, then **Runtime → Disconnect and delete runtime**.